# Challenge 2—Resilient Food Equity & Surplus Broker

### Load & Explore

Welcome. By the end of this notebook you will have four tables in your own BigQuery project,
you will know exactly which parts of them are real and which parts we generated, and you will
understand the one problem your team is going to spend the rest of the day solving.

**Read the text, not just the code.** This notebook is written so you can follow what is
happening by reading only the narrative between the cells. If you have never used BigQuery
before, you will still be able to follow. If you have, skim the prose and run the cells.

---

## What your team is building today

**A broker.** Somewhere in your city, a grocer, a farm, or a distributor has a pallet of
something perishable and a few hours to move it. Somewhere else, a pantry, a shelter, or a
clinic could use exactly that—but only if they have the right storage, serve the right
people, and can collect it in time.

Your agent sits between them. A coordinator describes what has just become available, and
the agent answers: **who should get this, why them, what can they actually take, and here is
the message to send them.**

The hard part is that this is not a lookup. *"10 crates of spinach, harvested yesterday,
needs cold storage"* has to find *"we serve 200 families, we have refrigeration, we need
fresh greens"*—and those two sentences share almost no words.

## Before anything else: which half of this data is real

You are going to be asked this by a judge, so let's settle it in the first thirty seconds.

**The organizations are real.** Every recipient in your corpus is a genuine tax-exempt
organization pulled live from the IRS, with its real name, real address, and real
classification. We do not invent charities.

**Their operational details are generated, and they had to be.** Whether a pantry has a
walk-in cooler, how many pallets it can take, when it can collect—**none of that is public
for any organization in the United States.** It is not hidden. Nobody has ever collected it
centrally. So we generate it, from archetypes, with the phrasing modelled on real published
profiles from Seattle and Pennsylvania.

That split is not a compromise we are apologising for. It is the honest shape of this
problem, and being able to say which half is which is part of what you are scored on.
Sections 5 through 7 show you exactly how the generated half is built, so you can inspect
it rather than take our word for it.

## What this notebook does

It gets you data, and it is honest with you about where every column came from. It does
**not** build your matcher and it does **not** build your agent. Those are yours.

Your required differentiating technology is **BigQuery vector search**. Section 14 sets up
the problem and hands you the syntax traps so you do not lose forty minutes to a function
name. It stops there deliberately—what to embed, how to compose a query, and what to do
with the ranked list are the decisions you are being judged on.

## Where this fits in your day

Your table is eight to ten people, which is too many to have everyone typing into one file.
The work splits into four lanes that run in parallel:

| Lane | What they do | Starts |
|---|---|---|
| **Data** | This notebook, then the embeddings and the match query | now |
| **Agent** | ADK agent, tools, the MCP server | now |
| **Front end** | The face: custom UI, Gemini Enterprise, or `adk web` | now, against a mock |
| **Story** | The match artifact, the pitch, the demo | now |

If you are reading this, you are probably the data lane. **Budget about 40 minutes.** The
single most useful thing you can do for your team is finish this and get the embeddings
built early, because the agent lane cannot call a search that does not exist yet.

# 1. Setup

## First, a decision that is not really a configuration setting

The cell below asks you to set `METRO`, and it is worth ten seconds of thought rather than
leaving the default.

**This is not a universal application. It is an application for one city.** Food rescue is
intensely local: a pallet has to physically move across town in a few hours, so a broker in
Dallas and a broker in Philadelphia are solving the same problem with completely different
organizations, different geography, and different neighbourhoods in need. Everything
downstream of this line—your recipients, your census tracts, your demo, your story—lives in
whichever city you pick.

**You do not have to pick the city you are sitting in.** Pick one your team can tell a story
about. If two of you know Chicago, build Chicago.

Two of the metros have a bonus: **Seattle and Philadelphia publish real operational profiles**
for their food programs, so if you pick either you can compare our generated text against the
genuine article in Section 6. That is a nice thing to be able to show a judge.

## What else we are about to do

Work out which Google Cloud project we are in, create a dataset to hold our tables, and set
up a timer so you can see where the minutes went.

The other line worth setting deliberately is `EMBED_ENDPOINT`.

**A word on `EMBED_ENDPOINT`.** BigQuery can call several different embedding models, and
they are not interchangeable—they return vectors of different sizes, and vectors from
different models live in different coordinate spaces and cannot be compared with each other.
Pick one at the start and stay on it. If you switch models later, you must re-embed
everything.

In [ ]:
# --- Configuration -----------------------------------------------------------

# WHICH CITY ARE YOU SOLVING FOR?
#
# This is a real decision, not a config value - see the markdown above.
# It sets which state's organizations we pull and which census tracts we load.
# You do NOT have to use the city you are sitting in.
#
# Tested end to end:
#     "Dallas"        [TX - the default]
#     "Seattle"       [WA - also has REAL published profiles to compare against, Section 6]
#     "Philadelphia"  [PA - also has REAL published profiles to compare against, Section 6]
#     "Atlanta"       [GA]        "Chicago"  [IL]
#     "Houston"       [TX]        "Denver"   [CO]
#     "New York"      [NY]
METRO = "Dallas"

# The embedding model your whole team will use. Stay on one.
#   'gemini-embedding-2'   3072 dims - newest and noticeably better at this task.
#                                     Verified working in BigQuery, 2026-08-07.
#   'text-embedding-005'    768 dims - older, still fine, and what most of Google's
#                                     published examples use.
#   'gemini-embedding-001' 3072 dims - accepts only ONE input text per request, so
#                                     embedding 500 profiles means 500 calls. Avoid in bulk.
EMBED_ENDPOINT = "gemini-embedding-2"

DATASET  = "a4i_food"     # if you change this, also change the %%bigquery cells that
                          # name it literally (Sections 3 and 12)
LOCATION = "US"           # keep every table in the same location as your connection

# How many recipient organizations to build a profile for. Big enough that reading
# them all is impractical - which is the entire reason semantic search earns its place.
TARGET_RECIPIENTS = 500

# --- Timing, so you know where your minutes went ------------------------------
import time
NOTEBOOK_START = time.time()
STEP_TIMES = {}

class step:
    """Context manager that times a block and reports it."""
    def __init__(self, name):
        self.name = name
    def __enter__(self):
        self.t0 = time.time()
        return self
    def __exit__(self, *exc):
        elapsed = time.time() - self.t0
        STEP_TIMES[self.name] = elapsed
        print(f"\n[{self.name}] took {elapsed:.1f}s")
        return False

# --- Which project are we in? -------------------------------------------------
import os
import google.auth

credentials, PROJECT_ID = google.auth.default()
if not PROJECT_ID:
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT")
if not PROJECT_ID:
    raise RuntimeError(
        "Could not determine your project ID. Set it manually:\n"
        "    PROJECT_ID = 'your-project-id-here'"
    )

# Metro -> (state postal code, bounding box) for the tract and organization pulls.
# Boxes are deliberately generous; we trim by tract intersection later.
METROS = {
    "Dallas":       ("TX", 32.60, 33.05,  -97.05,  -96.55),
    "Seattle":      ("WA", 47.20, 47.86, -122.55, -121.98),
    "Philadelphia": ("PA", 39.85, 40.15, -75.29,  -74.95),
    "Atlanta":      ("GA", 33.55, 33.92, -84.55,  -84.28),
    "Chicago":      ("IL", 41.64, 42.03, -87.94,  -87.52),
    "Houston":      ("TX", 29.52, 30.11, -95.79,  -95.01),
    "Denver":       ("CO", 39.61, 39.91, -105.11, -104.60),
    "New York":     ("NY", 40.49, 40.92, -74.26,  -73.70),
}
if METRO not in METROS:
    raise ValueError(f"Unknown metro {METRO!r}. Choose one of: {sorted(METROS)}")

STATE, LAT_MIN, LAT_MAX, LON_MIN, LON_MAX = METROS[METRO]

print(f"Project  : {PROJECT_ID}")
print(f"Dataset  : {DATASET} ({LOCATION})")
print(f"Metro    : {METRO}, {STATE}")
print(f"Embedding: {EMBED_ENDPOINT}")

Now we create the dataset. This is the one piece of setup that writes anything.

`exists_ok=True` means you can run this cell as many times as you like—useful, because
eight people sharing one project will run it more than once.

**The location matters more than it looks.** Later on you will create embeddings, and
BigQuery requires the connection doing that work to live in the same region as the dataset.
A dataset in `US` with a connection in `us-central1` fails with an error that does not
mention regions at all. Keep everything in one place.

In [ ]:
from google.cloud import bigquery

client = bigquery.Client(project=PROJECT_ID)

dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{DATASET}")
dataset_ref.location = LOCATION
dataset = client.create_dataset(dataset_ref, exists_ok=True)

print(f"Ready: {dataset.full_dataset_id}  (location: {dataset.location})")

# 2. Start with the wrong answer

## Why we are doing this on purpose

The obvious way to match a food donation to an organization is to search for words. The
donation says "spinach," so look for organizations that mention spinach.

Let's watch that fail twice, in two different datasets, in about ninety seconds. It is the
fastest way to understand what the rest of the day is for.

**Failure one: the recipients.** Here are three real profile sentences, written the way real
organizations actually write them. Ask yourself which one should receive ten crates of
spinach that need refrigeration—then look at how many words the winning answer shares with
the request.

In [ ]:
offer = "10 crates of spinach, harvested yesterday, needs cold storage"

profiles = {
    "Northside Pantry":       "we serve 200 families, we have refrigeration, we need fresh greens",
    "Riverside Shelter":      "hot meals nightly, no cold storage, need shelf-stable goods",
    "Eastside Family Clinic": "infant formula and diapers, temperature controlled storage only",
}

offer_words = set(offer.lower().replace(",", "").split())

print(f"OFFER: {offer}\n")
print(f"{'organization':<24} {'shared words':<14} words in common")
print("-" * 72)
for name, text in profiles.items():
    shared = offer_words & set(text.lower().replace(",", "").split())
    interesting = {w for w in shared if len(w) > 3}
    print(f"{name:<24} {len(interesting):<14} {sorted(interesting) or 'none'}")

## Look at what came back

The right answer is Northside Pantry. It has refrigeration and it wants fresh greens.

And it shares **no meaningful words at all** with the request. "Spinach" is not "fresh
greens." "Cold storage" is not "refrigeration." A keyword search does not rank it first—a
keyword search does not find it.

Worse, Riverside Shelter contains the exact phrase *"cold storage"*—as part of **"no cold
storage."** Keyword matching scores it highly for the precise reason it should be excluded.

**Failure two: the food itself.** Hold that thought. In Section 9 we load USDA's shelf-life
data to work out how long the spinach has. That dataset has 661 products in it, and **not one
of them is called "Spinach."** You will see what it is called instead, and why an exact-name
lookup for the most obvious search term in this challenge returns nothing.

Same lesson, two datasets, before you have written a line of agent code.

## The shape of the fix

What you need is a way to compare *meaning* rather than *characters*. BigQuery can do this
directly. Run the cell below—it is the whole idea in one statement, and it needs no setup,
no model, and no stored data.

In [ ]:
# Run this through the Python client rather than the %%bigquery magic, so it can
# use the EMBED_ENDPOINT you set in Section 1 instead of a hardcoded model name.
sim_sql = f"""
SELECT
  organization,
  ROUND(AI.SIMILARITY(
    content1 => @offer,
    content2 => profile,
    endpoint => '{EMBED_ENDPOINT}'), 4) AS similarity
FROM UNNEST([
  STRUCT('Northside Pantry'       AS organization,
         'we serve 200 families, we have refrigeration, we need fresh greens' AS profile),
  STRUCT('Riverside Shelter',
         'hot meals nightly, no cold storage, need shelf-stable goods'),
  STRUCT('Eastside Family Clinic',
         'infant formula and diapers, temperature controlled storage only')
])
ORDER BY similarity DESC
"""

cfg = bigquery.QueryJobConfig(query_parameters=[
    bigquery.ScalarQueryParameter("offer", "STRING", offer)])

with step("semantic similarity"):
    sim = client.query(sim_sql, job_config=cfg).to_dataframe()

print(f"OFFER: {offer}")
print(f"MODEL: {EMBED_ENDPOINT}\n")
print(f"{'ORGANIZATION':<26}{'SIMILARITY':>11}")
print("-" * 50)
for rank, row in enumerate(sim.itertuples()):
    flag = "   <-- best match" if rank == 0 else ""
    print(f"{row.organization:<26}{row.similarity:>11.4f}{flag}")
print("-" * 50)
print(f"\nBest semantic match: {sim.iloc[0].organization}")
print(f"Best keyword match : Riverside Shelter (it contains the exact phrase 'cold storage')")

**Northside Pantry should be at the top**, and the gap matters as much as the order. It wins
on meaning alone, without sharing a single useful word with the request.

Look at where Riverside Shelter landed. It is the one organization that literally contains the
phrase *"cold storage"*—and a keyword search would have ranked it first for exactly the reason
it should be excluded, because the full phrase is *"no cold storage."* Semantic matching puts
it second, behind the organization that never says "cold storage" at all.

You have just seen the entire premise of this challenge work in one query, with zero setup.

**So why not stop here?** Because `AI.SIMILARITY` embeds both sides *every time you call it*.
At three candidates that is fine. At five hundred, on every request, it is slow and wasteful—you would be re-computing the same five hundred vectors over and over.

What you actually want is to embed the organizations **once**, store the vectors, and search
them. That is `VECTOR_SEARCH`, and it is your differentiator. Section 14 sets it up.

Now let's go and get five hundred real organizations.

# 3. The recipients are real, and they come from the IRS

## Why this dataset

We looked hard for a public list of food pantries with operational detail, and the honest
answer is that the good ones are all locked up. The two largest directories in the United
States—one with 16,697 listings, one with 8,300—are both "all rights reserved," and one
of them explicitly forbids reproduction. Feeding America publishes nothing machine-readable.
New York City and Chicago, the two open-data portals you would most expect to have this,
simply do not publish a pantry directory.

But there is one national list that is genuinely public, and it is a good one.

Every tax-exempt organization in the United States appears in the IRS **Exempt Organizations
Business Master File**—1.98 million of them, refreshed monthly, published as plain CSV by
state. It carries each organization's name, street address, city, ZIP, and—the part that
makes it useful here—an **NTEE code** classifying what the organization actually does.

### What is an NTEE code, and do you need to care?

**Short answer: not really, and you can skip this box.** It is the classification system the
IRS uses to record what a nonprofit actually does—a letter for the broad field (`K` is Food,
Agriculture and Nutrition; `L` is Housing and Shelter; `E` is Health) and digits for the
specific activity. It exists so that "Community Kitchen Inc." and "St. Anne's Pantry" can be
recognised as the same kind of organization without reading their names.

All you need from it is this: **it is the only reliable way to find food-capable organizations
in a file of two million charities.** Filtering on the word "food" in the name would miss
every church pantry and catch every restaurant association.

These are the codes we keep:

| Code | What it means |
|---|---|
| `K30` | Food service, free food distribution programs |
| `K31` | **Food banks and food pantries**—the biggest group by far |
| `K34` | Congregate meals and nutrition sites |
| `K35` | Soup kitchens |
| `K36` | Home-delivered meals |
| `L41` | Temporary shelter |
| `E32` | Community and ambulatory health clinics |

If you would rather widen or narrow that list later, it is one tuple in the next code cell.
That is a legitimate design decision and a judge would be interested in why you changed it.

**Licence:** this is a work of the United States federal government, in the public domain
under 17 U.S.C. § 105. There is no licence page to point at because federal works do not
need one.

## One thing before we download

We are about to fetch a file directly from `irs.gov`. Some government sites reject requests
that do not come from a browser—you will meet one of those in Section 9—so we set a
user-agent header and we retry. If a hundred people in this room hit the same server in the
same three minutes, one of you will get a connection reset. That is not your fault and it is
not a bug in your code.

In [ ]:
import io
import time as _time
import requests
import pandas as pd

EO_URL = f"https://www.irs.gov/pub/irs-soi/eo_{STATE.lower()}.csv"

# Codes that identify an organization able to receive and distribute food or essentials.
NTEE_PREFIXES = ("K30", "K31", "K34", "K35", "K36", "L41", "E32")


def fetch_with_retry(url, tries=4, timeout=120):
    """Government endpoints are inconsistent under load. Back off and try again."""
    headers = {"User-Agent": "Mozilla/5.0 (compatible; A4I-2026-workshop)"}
    last = None
    for attempt in range(tries):
        try:
            r = requests.get(url, headers=headers, timeout=timeout)
            r.raise_for_status()
            return r
        except Exception as exc:                      # noqa: BLE001
            last = exc
            wait = 2 ** attempt
            print(f"  attempt {attempt + 1} failed ({exc}); retrying in {wait}s")
            _time.sleep(wait)
    raise RuntimeError(f"Could not fetch {url} after {tries} attempts: {last}")


with step("download IRS organizations"):
    print(f"Fetching {EO_URL}")
    resp = fetch_with_retry(EO_URL)
    eo_all = pd.read_csv(io.BytesIO(resp.content), dtype=str, low_memory=False)

print(f"\nAll tax-exempt organizations in {STATE}: {len(eo_all):,}")
print(f"Columns as delivered: {list(eo_all.columns)}")

**Print the columns before you trust them.** We just did, and you should look at what came
back rather than at what we told you to expect. Column names in public datasets drift, and a
notebook that hardcodes them breaks silently six months later.

Now the same discipline applied to values. We are about to filter on `NTEE_CD`, and it turns
out the codes in this file are **both three and four characters**—you will see `K31` and
`K310` and `K31Z` side by side, all meaning food bank or pantry.

That is exactly the kind of thing that is easy to assume and expensive to get wrong. So rather
than guessing, ask the data.

In [ ]:
# What do the NTEE codes in this file actually look like?
ntee = eo_all["NTEE_CD"].dropna().astype(str)
print(f"Non-null NTEE codes: {len(ntee):,}")
print(f"Distinct lengths   : {sorted(ntee.str.len().unique())}")
print("\nA sample of codes starting with K3 (our food codes):")
print(sorted(ntee[ntee.str.startswith("K3")].unique())[:20])

That is why we match on a **prefix** rather than an exact value. Whatever the trailing
characters turn out to be—`0`, `Z`, `C`, `I`, or nothing at all—`K31` at the front means food
bank or pantry.

You will also see codes in that list we do *not* keep, like `K32` and `K33Z`. They are
adjacent activities in the same family. Leaving them out is a judgment call rather than a
rule, and widening the net is a perfectly reasonable thing for your team to do.

Now the filter.

In [ ]:
with step("filter to food-capable organizations"):
    mask = eo_all["NTEE_CD"].fillna("").str.upper().str.startswith(NTEE_PREFIXES)
    eo = eo_all[mask].copy()

    # STATUS 01 = currently recognised as exempt. Everything else is lapsed,
    # revoked, or pending, and we do not want an agent emailing a defunct charity.
    if "STATUS" in eo.columns:
        before = len(eo)
        eo = eo[eo["STATUS"].fillna("").str.strip() == "01"]
        print(f"Dropped {before - len(eo):,} organizations not currently exempt (STATUS != 01)")

print(f"\nFood-capable organizations in {STATE}: {len(eo):,}")
print("\nBy NTEE code:")
print(eo["NTEE_CD"].str[:3].value_counts().head(10).to_string())

# 4. Two columns we delete on sight, and why

This is the most important cell in the first half of the notebook, and it has nothing to do
with vectors.

You have just loaded a public dataset with a clean licence. That tells you that you are
**allowed** to use it. It tells you nothing about whether every column in it **belongs** in
the application you are about to build.

Two columns here do not.

**`ICO`—"In Care Of".** For many small organizations this is a person's name. A private
individual, published in a federal file because they volunteered to be a point of contact
for a church pantry in 2003. Nothing in the licence stops you loading it. Load it anyway and
your agent will eventually put a stranger's name in a generated email.

**`STREET`** is more subtle. For a regional food bank it is a warehouse. For a two-person
pantry it is frequently **somebody's house**, because that is where the operation runs from.
The column is not labelled "sometimes a home address." It just is one, sometimes.

We drop `ICO` entirely. We keep `STREET`, because a broker genuinely needs to know where to
send a truck—but we keep it knowing what it sometimes is, which is a different thing from
keeping it carelessly.

> **The rule worth taking with you:** "we checked the licence" and "we checked what is in it"
> are two separate pieces of work. Judges will ask about the second one. Most teams will only
> have done the first.

In [ ]:
KEEP = ["EIN", "NAME", "STREET", "CITY", "STATE", "ZIP", "NTEE_CD", "INCOME_AMT", "REVENUE_AMT"]

dropped = [c for c in eo.columns if c not in KEEP]
recipients = eo[[c for c in KEEP if c in eo.columns]].copy()

print(f"Kept   : {list(recipients.columns)}")
print(f"Dropped: {len(dropped)} columns, including ICO (a person's name for many orgs)")
assert "ICO" not in recipients.columns, "ICO must not survive into the loaded table"

# ZIP arrives as a string and may carry the +4 extension. Normalise to five digits.
recipients["ZIP5"] = recipients["ZIP"].fillna("").astype(str).str.strip().str[:5]

# Money columns are strings in this file. SAFE-cast rather than trusting them.
for col in ("INCOME_AMT", "REVENUE_AMT"):
    if col in recipients.columns:
        recipients[col] = pd.to_numeric(recipients[col], errors="coerce").fillna(0)

# Titlecase the shouting. The IRS publishes names in all caps.
recipients["NAME"] = recipients["NAME"].fillna("").str.title()

print(f"\nOrganizations carried forward: {len(recipients):,}")
recipients.head(8)[["EIN", "NAME", "CITY", "ZIP5", "NTEE_CD", "INCOME_AMT"]]

# 5. What the IRS does not tell you

We have real organizations with real names and real addresses. Now look at what we need in
order to broker a pallet of spinach, and notice how little of it we have:

| To make a match we need to know | Do we have it? |
|---|---|
| Does this organization exist and what is it | ✅ from the IRS |
| Where is it | ✅ from the IRS |
| Roughly how big is it | ✅ `INCOME_AMT` is a decent proxy |
| **Do they have refrigeration** | ❌ |
| **Can they take a pallet, or a bag** | ❌ |
| **Who do they serve** | ❌ |
| **When are they open, and can they collect** | ❌ |
| **Is there anything they cannot accept** | ❌ |

Every row in the second half of that table is the information a broker actually decides on,
and **none of it is public for any organization in the country.** It is not hidden—it is
simply not collected anywhere central. A pantry knows whether it has a walk-in cooler. Nobody
has ever asked all of them at once.

So we have a choice, and it is worth being explicit about it because a judge will ask.

We could pretend the problem away and match on category alone—"this is a food bank, food
banks take food." That produces an agent that cannot tell the difference between a warehouse
with a freezer and a card table in a church hall, which is the entire decision.

Or we can generate the operational half, from real examples, and say so plainly.

We are doing the second thing. The next two sections show you exactly how, because **you
should not trust generated data that you cannot inspect**—and neither should the judges.

# 6. First, what a real profile actually sounds like

Two places in the United States publish genuine operational detail for their food programs
under a licence we can use: **Seattle / King County** (Public Domain) and **Pennsylvania**
(US Government Works). Between them that is about 550 organizations.

That is not enough to build a national challenge on. But it is more than enough to answer the
question that matters before generating anything: *what does this kind of text look like when
a human who runs a pantry writes it?*

Let's go and read some.

In [ ]:
SEATTLE_URL = "https://cos-data.seattle.gov/resource/kkzf-ntnu.json?$limit=500"

real_profiles = pd.DataFrame()
try:
    with step("fetch real operational profiles"):
        r = fetch_with_retry(SEATTLE_URL, tries=3, timeout=60)
        real_profiles = pd.json_normalize(r.json())
    print(f"Real Seattle/King County records: {len(real_profiles):,}")
    print(f"Columns: {sorted(real_profiles.columns)}")
except Exception as exc:                              # noqa: BLE001
    print(f"Could not reach the Seattle open-data API ({exc}).")
    print("Not fatal - this section is illustrative. Continuing.")

Look at the `operational_status` field in particular. This is what real operational text
looks like, and it is nothing like a database record.

In [ ]:
if len(real_profiles):
    shown = 0
    for _, row in real_profiles.iterrows():
        note = str(row.get("operational_status", "") or "").strip()
        if len(note) < 60:
            continue
        print("=" * 78)
        print(f"{row.get('agency', '?')}  [{row.get('food_resource_type', '?')}]")
        print(f"  serves : {row.get('who_they_serve', '?')}")
        print(f"  days   : {row.get('days_hours', '?')}")
        print(f"  notes  : {note[:400]}")
        shown += 1
        if shown >= 5:
            break
    print("=" * 78)
else:
    print("No real profiles loaded - see the previous cell.")

Read those carefully, because three things in them shape everything that follows.

**Constraints arrive as prose, not as fields.** *"Open for Seniors only on Wednesdays
2:00pm-3:00pm, Thursdays 10:00am-12:00pm."* No database schema would have predicted that
shape. A structured form would have flattened it into something less true.

**The same capability is described in wildly different words.** Across these records you will
find refrigeration described as cold storage, a walk-in, a cooler, and simply "we have a
fridge." That is not sloppiness—that is how people write. And it is precisely why keyword
matching failed in Section 2.

**Some of what is here is a hard operational constraint.** *"Serves residents of the Auburn
School District."* *"Not required to show proof of residence."* A broker that ignores these
sends a truck that gets turned away.

Now we generate the rest—in that voice.

# 7. Generating the operational half, honestly

## Why we are about to do something that might look like cheating

We are going to invent data. In a challenge about public data. That deserves a straight
explanation before a single line of it exists, because if you cannot defend this in your demo
it will read as a shortcut instead of a decision.

**The alternative was worse.** We could have matched on category alone—"this is a food bank,
food banks take food." That builds an agent that cannot tell a warehouse with a walk-in
freezer from a card table in a church hall, which is the entire decision a broker makes. It
would have been technically honest and practically useless.

**We could not find the real thing, and we looked hard.** The two largest pantry directories
in the country are both all-rights-reserved and one explicitly forbids reproduction. Feeding
America publishes nothing machine-readable. New York City and Chicago do not publish a pantry
directory at all. Only Seattle and Pennsylvania do it openly—about 550 organizations between
them, which is not a national challenge.

**So the split is: real organizations, generated capabilities.** And here is the sentence
worth memorising for when a judge asks you where the data came from:

> *"The organizations are real—IRS Exempt Organizations file, federal public domain, filtered
> by NTEE classification. The operational details are not public for any pantry in the
> country, so we generated them from archetypes modelled on real published profiles from
> Seattle and Pennsylvania. Generation is deterministic and the vocabulary banks are in the
> repo."*

That answer beats "we made some data up," and it beats pretending the whole thing is real.
**A team that knows which half is which is demonstrating exactly the judgment this challenge
rewards.**

Everything below shows you how it is built, so you can check rather than trust.

## The rules we set ourselves

**One: the organizations stay real.** We never invent a name or an address. Every profile
below is attached to an organization that genuinely exists and is genuinely classified as a
food or shelter provider by the IRS.

**Two: generation is deterministic.** Each organization's profile is seeded from its EIN.
The same organization produces the same profile on every run, for every team, in every city.
That matters for a reason that is easy to miss—if the corpus shifted every time somebody
re-ran the notebook, your validation checks would be meaningless and no two teammates would
be debugging the same data.

**Three: archetypes, not a template.** This is the part that actually decides whether your
vector search works. If we generated five hundred profiles from one pattern, they would all
land in nearly the same place in embedding space, and the search would return five hundred
near-ties in effectively random order. It would *look* like it was working. It would be a
coin flip.

So instead there are fourteen archetypes—a regional warehouse, a church basement, a mobile
van, a clinic, a diaper bank, a rural outpost—each with its own vocabulary, its own
constraints, and its own way of writing. An organization's archetype is chosen from its NTEE
code and its income, so a large `K31` becomes a warehouse and a small one becomes a pantry.

**Four: we plant the hard cases on purpose.** More on this after the code.

In [ ]:
import hashlib
import json
import random
from pathlib import Path

# The vocabulary banks ship with the repo so you can read exactly what we used.
COMPONENTS_PATH = Path("../data/profile_components.json")
if not COMPONENTS_PATH.exists():
    COMPONENTS_PATH = Path("data/profile_components.json")

if COMPONENTS_PATH.exists():
    COMPONENTS = json.loads(COMPONENTS_PATH.read_text())
    print(f"Loaded vocabulary banks from {COMPONENTS_PATH}")
else:
    # Colab Enterprise imports a single notebook, not the repo. Fall back to the raw URL.
    RAW = ("https://raw.githubusercontent.com/haggman/"
           "A4I2026-challenge-2-food-equity/main/data/profile_components.json")
    COMPONENTS = fetch_with_retry(RAW, tries=3, timeout=60).json()
    print(f"Loaded vocabulary banks from {RAW}")

ARCHETYPES = COMPONENTS["archetypes"]
print(f"\n{len(ARCHETYPES)} archetypes available:")
for key, a in ARCHETYPES.items():
    print(f"  {key:<20} {a['label']}")

Now the assignment rule. Notice that it is boring on purpose—NTEE code plus income
bracket. A judge should be able to read this in ten seconds and agree it is reasonable.

In [ ]:
def choose_archetype(ntee, income, rng):
    """Pick an archetype from what the IRS actually tells us about this organization."""
    code3 = (ntee or "")[:3].upper()
    big = income >= 1_000_000
    small = income < 100_000

    if code3 == "E32":
        return "community_clinic"
    if code3 == "L41":
        return rng.choice(["shelter_kitchen", "shelter_kitchen", "disaster_staging"])
    if code3 == "K35":
        return "soup_kitchen"
    if code3 in ("K34", "K36"):
        return "senior_meals"
    if code3 == "K30":
        if big:
            return rng.choice(["regional_foodbank", "produce_coop"])
        return rng.choice(["mobile_pantry", "produce_coop", "diaper_bank"])
    # K31 - the big bucket: food banks and pantries
    if big:
        return "regional_foodbank"
    if small:
        return rng.choice(["neighborhood_pantry", "neighborhood_pantry", "faith_weekly",
                           "school_pantry", "halal_kosher", "rural_outpost"])
    return rng.choice(["neighborhood_pantry", "mobile_pantry", "faith_weekly", "senior_meals"])


def build_profile(row):
    """Deterministically synthesize one organization's operational profile.

    Seeded from the EIN, so this organization always gets this profile.
    """
    seed = int(hashlib.sha256(str(row["EIN"]).encode()).hexdigest()[:12], 16)
    rng = random.Random(seed)

    key = choose_archetype(row.get("NTEE_CD"), float(row.get("INCOME_AMT") or 0), rng)
    a = ARCHETYPES[key]

    households = rng.randint(*a["weekly_households"])
    sites = rng.randint(3, 22)
    sqft = rng.choice([8000, 12000, 18000, 24000, 40000])

    opener = rng.choice(a["openers"]).format(n=sites, h=households, sqft=f"{sqft:,}")
    storage = rng.choice(a["storage"])
    logistics = rng.choice(a["logistics"])
    needs = rng.choice(a["needs"])
    serves = rng.choice(COMPONENTS["populations"])
    days = rng.choice(COMPONENTS["service_days"])
    suffix = rng.choice(COMPONENTS["notes_suffix"])

    text = " ".join(p for p in [opener, storage, logistics, needs,
                                f"Open {days}.", suffix] if p).strip()

    return pd.Series({
        "archetype":              key,
        "archetype_label":        a["label"],
        "cold_storage":           a["cold_storage"],
        "frozen_storage":         bool(a["frozen"]),
        "loading_dock":           bool(a["dock"]),
        "refrigerated_transport": bool(a["refrigerated_transport"]),
        "max_pallets":            rng.randint(*a["max_pallets"]),
        "pickup_window_hours":    rng.randint(*a["pickup_window_hours"]),
        "weekly_households":      households,
        "populations_served":     serves,
        "service_days":           days,
        "profile_text":           text,
        "profile_source":         "generated",
    })


with step("generate operational profiles"):
    # Deterministic sample: sort by EIN so the same organizations are chosen every run.
    pool = recipients.sort_values("EIN").reset_index(drop=True)
    if len(pool) > TARGET_RECIPIENTS:
        pool = pool.iloc[:TARGET_RECIPIENTS].copy()

    generated = pool.join(pool.apply(build_profile, axis=1))

print(f"Profiles generated: {len(generated):,}")
print("\nArchetype spread - a healthy corpus is spread, not concentrated:")
print(generated["archetype"].value_counts().to_string())

## The hard cases, and why they are in there

A generated corpus where every profile is clean and every question has one obvious answer
teaches nothing and proves nothing. Four kinds of difficulty are planted deliberately:

**Vocabulary mismatch.** Refrigeration is described five different ways across the corpus—"cold chain capacity," "walk-in cooler," "we have a fridge," "refrigerated storage,"
"chilled holding." If your matcher works on these, it is working on meaning.

**A near-miss decoy.** One organization *wants* fresh greens more than anything, and has **no
refrigeration at all.** A keyword match ranks it first. A semantic match probably ranks it
first too—because it genuinely is about fresh greens. **Only the storage constraint catches
it.** This is the single most useful row in the corpus, because it teaches that vector search
is a *candidate generator*, not an answer. What you do after the search is where your agent
earns its score.

**A contradiction.** One profile claims a walk-in freezer in one sentence and a single shelf
in a shared cupboard in the next. Real data contains contradictions. Yours should too, so
that you find out whether you noticed.

**Two genuinely tied answers.** Two organizations are equally correct for the same offer. A
coordinator would have to choose and justify it. So should your agent—"these two are
equivalent on capability, we chose this one because..." is a much better demo moment than a
confident number.

In [ ]:
planted = COMPONENTS["planted_cases"]
rows = []


def _plant(name, text, **over):
    base = {
        "EIN": f"P{len(rows):08d}", "NAME": name, "STREET": "", "CITY": METRO,
        "STATE": STATE, "ZIP5": "", "NTEE_CD": "K31", "INCOME_AMT": 0, "REVENUE_AMT": 0,
        "archetype": "planted", "archetype_label": "Planted teaching case",
        "cold_storage": "none", "frozen_storage": False, "loading_dock": False,
        "refrigerated_transport": False, "max_pallets": 1, "pickup_window_hours": 12,
        "weekly_households": 200, "populations_served": "General public",
        "service_days": "Monday, Wednesday, Friday",
        "profile_text": text, "profile_source": "planted",
    }
    base.update(over)
    rows.append(base)


d = planted["near_miss_decoy"]
_plant("Greenline Community Pantry", d["profile"], cold_storage="none", max_pallets=1)

c = planted["contradiction"]
_plant("Harborview Neighbors Trust", c["profile"], cold_storage="walk-in",
       frozen_storage=True, max_pallets=6)

for i, p in enumerate(planted["genuine_ties"]["profiles"]):
    _plant(f"{'Cedar' if i == 0 else 'Maple'} Street Produce Alliance", p,
           cold_storage="walk-in", refrigerated_transport=True, loading_dock=True,
           max_pallets=10, weekly_households=600)

for i, phrasing in enumerate(planted["vocabulary_mismatch"]):
    _plant(f"Vocabulary Test Site {i + 1}",
           f"Neighborhood food distribution serving local families. {phrasing} "
           f"We take fresh produce weekly. Open Tuesday, Thursday.",
           cold_storage="walk-in" if i < 4 else "none", max_pallets=3)

recipients_final = pd.concat([generated, pd.DataFrame(rows)], ignore_index=True)
recipients_final["recipient_id"] = range(1, len(recipients_final) + 1)

print(f"Corpus: {len(recipients_final):,} organizations "
      f"({(recipients_final.profile_source == 'generated').sum():,} generated, "
      f"{(recipients_final.profile_source == 'planted').sum():,} planted)")
print("\n--- one example profile ---\n")
ex = recipients_final.iloc[7]
print(f"{ex['NAME']}  [{ex['archetype_label']}]")
print(ex["profile_text"])

## Inspect it before you trust it

Everything the generator can possibly say about an organization is in
`data/profile_components.json`—fourteen archetypes, their storage and transport
characteristics, and every phrase they draw from. It is worth two minutes of reading.

**You should not trust generated data you cannot inspect, and neither should a judge.** Being
able to open the file and show what is in it is the difference between "we generated profiles"
and "we engineered a corpus."

Section 13 then validates the result—including the one number that decides whether any of this
works: how many of the profiles are actually distinct from each other.

# 8. The clock: how long does the food have?

A match is not just about who *wants* the spinach. It is about who can get it before it stops
being food. So we need a shelf life for whatever is on offer.

**USDA's FoodKeeper dataset** gives us that: 661 products with shelf life in the pantry, the
refrigerator, and the freezer. It is **CC0**—public domain, no attribution required—and it has
not changed since 2018, which makes it about the most stable thing in this challenge.

## Why this file is in the repo instead of being downloaded

Here is a real-world lesson that cost us an afternoon, and it is worth more than the data.

FoodKeeper is published by USDA at two government URLs. Both of them return a perfectly good
file **in your browser** and **403 Forbidden** to this notebook. Not because of anything you
did—because federal sites sit behind bot protection that rejects requests from datacenter IP
ranges, and every Colab runtime on Earth lives in a datacenter.

We tried the obvious fixes: browser-shaped headers, a session cookie from the parent page,
both hosts. All 403.

**So the repo carries its own copy.** The file is public domain, it is 630 KB, and it has not
changed in eight years. Pinning it removes an event-day dependency on a server that can decide
it does not like us, on a morning when 150 people hit it at once.

> **The generalisable version:** "the data is public" and "I can fetch the data from my code"
> are different claims. Check the second one early, from the environment you will actually run
> in. A licence check is not a connectivity check.

The cell below still tries the live sources first, so you can see for yourself and so the
notebook keeps working if USDA ever relaxes. It reports which one it used.

## And one thing about the file itself

The units are strings, and **four of their values are not units at all.** More on that once we
can see them.

In [ ]:
# FoodKeeper is CC0 public-domain USDA data, but both government hosts reject
# requests from datacenter IP ranges - which is where every Colab runtime lives.
# So the repo carries its own copy, and the live hosts are the fallback.

REPO_RAW = ("https://raw.githubusercontent.com/haggman/"
            "A4I2026-challenge-2-food-equity/main")

# NOTE ON THE CLOUD STORAGE URL, because this catches people:
#   storage.googleapis.com/<bucket>/<object>   <- the API endpoint. Anonymous,
#                                                 returns the file. Use this.
#   storage.cloud.google.com/<bucket>/<object> <- the CONSOLE endpoint. Needs a
#                                                 browser session; returns an HTML
#                                                 page to a script, with status 200.
# The second one looks like it works and hands you HTML instead of JSON.
GCS_MIRROR = ("https://storage.googleapis.com/class-demo/a4i-2026/"
              "challenge-2-food-equity/reference/foodkeeper.json")

FOODKEEPER_SOURCES = [
    ("repo copy",      f"{REPO_RAW}/data/foodkeeper.json"),
    ("ROI mirror",     GCS_MIRROR),
    ("FoodSafety.gov", "https://www.foodsafety.gov/sites/default/files/foodkeeper_data_url_en.json"),
]


def fetch_foodkeeper():
    """Try each known host. Return (json, source_name). Raise only if all fail."""
    problems = []
    for label, url in FOODKEEPER_SOURCES:
        try:
            r = requests.get(url, headers={"User-Agent": "A4I-2026-workshop"}, timeout=90)
            r.raise_for_status()
            print(f"  OK      {label}")
            return r.json(), label
        except Exception as exc:                          # noqa: BLE001
            code = getattr(getattr(exc, "response", None), "status_code", "")
            print(f"  no      {label}  {code} {type(exc).__name__}")
            problems.append(f"{label}: {exc}")
    raise RuntimeError(
        "No FoodKeeper source reachable.\n  " + "\n  ".join(problems) +
        "\n\nIf the repo copy 404s, your team's repository is missing "
        "data/foodkeeper.json - tell a coach.")


with step("download FoodKeeper"):
    print("Trying FoodKeeper sources in order:")
    fk_raw, FK_SOURCE = fetch_foodkeeper()

print(f"\nLoaded from: {FK_SOURCE}")
print(f"Top-level keys: {list(fk_raw.keys())}")
for k, v in fk_raw.items():
    if isinstance(v, list):
        print(f"  {k}: {len(v):,} records")

products = pd.DataFrame(fk_raw["product_data"])
print(f"\nProducts: {len(products):,}")
print(f"Columns : {sorted(products.columns)}")

## And here is the second half of Section 2's lesson

Remember the promise: there is no product called "Spinach" in this file. Let's prove it, and
then find where spinach actually lives.

In [ ]:
name_col = "name" if "name" in products.columns else "Name"

exact = products[products[name_col].fillna("").str.lower() == "spinach"]
print(f"Products named exactly 'Spinach': {len(exact)}")

print("\nSearching every text column for the word instead...")
hits = products[products.astype(str).apply(
    lambda r: "spinach" in " ".join(r.values).lower(), axis=1)]
print(f"Records mentioning spinach anywhere: {len(hits)}\n")

for _, row in hits.head(4).iterrows():
    sub = row.get("name_subtitle") or row.get("Name_subtitle") or ""
    print(f"  name = {row[name_col]!r:<28} subtitle = {str(sub)!r}")

There it is. Spinach is not a product—it is a **subtitle on "Lettuce."**

An exact-name lookup for the single most obvious search term in this entire challenge returns
zero rows, from a dataset that unambiguously contains the answer. That is the same failure
you saw in Section 2, in a completely different dataset, for a completely different reason.

**This is not a quirk of USDA's data.** It is what real reference data is like. The word you
have and the word the data uses are different words, and a system built on exact matching
breaks on contact with reality.

## Now the units problem

We want hours. The file gives us a number and a unit string. Let's look at every unit value
that actually appears before we write a conversion.

In [ ]:
# The mirror pre-renders durations as display strings like "1 - 2 Weeks".
disp = [c for c in products.columns if c.endswith("output_display_only")]
print(f"Duration columns: {len(disp)}")
for c in disp[:6]:
    print(f"  {c}")

import re
UNIT_RE = re.compile(r"(days?|hours?|weeks?|months?|years?|indefinitely|not recommended|"
                     r"package use-by date|when ripe)", re.I)

units = set()
for c in disp:
    for v in products[c].dropna().astype(str):
        units.update(m.lower() for m in UNIT_RE.findall(v))

print(f"\nDistinct units found: {sorted(units)}")

Look at that list. Alongside `days`, `weeks`, and `months` you have **`indefinitely`**,
**`not recommended`**, **`package use-by date`**, and **`when ripe`**.

Those are not durations. They are sentences that arrived in a numeric column.

If you write the obvious conversion—parse the number, multiply by the unit—those four
values become nulls, or worse, zeros. **A zero means "this food has already expired," and an
agent reading that will refuse to broker perfectly good tinned goods.** This is the same class
of error as a `-9999` sitting in a temperature column: it does not raise, it just quietly
makes your answers wrong.

We handle them explicitly, and visibly.

Also note `Year` **and** `Years` both appear. That is a real inconsistency in the source file,
not something we introduced.

In [ ]:
UNIT_HOURS = {
    "hour": 1, "hours": 1,
    "day": 24, "days": 24,
    "week": 168, "weeks": 168,
    "month": 720, "months": 720,
    "year": 8760, "years": 8760,          # both spellings appear in the source
}
SENTINELS = {
    "indefinitely":         ("shelf_stable", 24 * 365 * 2),
    "not recommended":      ("not_recommended", None),
    "package use-by date":  ("see_package", None),
    "when ripe":            ("ripeness_dependent", 72),
}


def parse_duration(text):
    """Return (hours, note). Sentinel strings get a note, never a silent zero."""
    if not text or str(text).strip() in ("", "nan", "None"):
        return None, "not_specified"
    s = str(text).strip().lower()
    for sentinel, (note, hours) in SENTINELS.items():
        if sentinel in s:
            return hours, note
    nums = [float(n) for n in re.findall(r"\d+(?:\.\d+)?", s)]
    unit = next((u for u in UNIT_RE.findall(s) if u.lower() in UNIT_HOURS), None)
    if not nums or not unit:
        return None, "unparsed"
    return max(nums) * UNIT_HOURS[unit.lower()], "ok"


def pick(row, *cands):
    for c in cands:
        if c in row and pd.notna(row[c]) and str(row[c]).strip():
            return row[c]
    return None


with step("parse shelf life"):
    recs = []
    for _, row in products.iterrows():
        fridge = pick(row, "refrigerate_output_display_only",
                      "from_date_of_purchase_refrigerate_output_display_only",
                      "refrigerate_after_opening_output_display_only")
        pantry = pick(row, "pantry_output_display_only",
                      "from_date_of_purchase_pantry_output_display_only",
                      "pantry_after_opening_output_display_only")
        fh, fnote = parse_duration(fridge)
        ph, pnote = parse_duration(pantry)
        recs.append({
            "product_id":          str(row.get("id") or row.get("ID")),
            "product_name":        row[name_col],
            "product_subtitle":    row.get("name_subtitle") or "",
            "keywords":            row.get("keywords") or "",
            "category":            row.get("category_name_display_only") or "",
            "refrigerated_hours":  fh,
            "refrigerated_note":   fnote,
            "pantry_hours":        ph,
            "pantry_note":         pnote,
        })
    shelf_life = pd.DataFrame(recs)

print(f"Products parsed: {len(shelf_life):,}")
print("\nWhat happened to the refrigerated column:")
print(shelf_life["refrigerated_note"].value_counts().to_string())
print("\nThe sparsity is real - most products simply do not carry every storage state.")
print(f"  with a refrigerated figure : {shelf_life.refrigerated_hours.notna().sum():,}")
print(f"  with a pantry figure       : {shelf_life.pantry_hours.notna().sum():,}")

# 9. Who lives around them

A broker that only asks "who can store this" will send every pallet to the biggest warehouse.
That is efficient and it is not the point. The reason this challenge is about *equity* is
that two organizations with identical refrigeration can sit in neighbourhoods with completely
different levels of need.

So we load the census. Three signals, all from the American Community Survey, all in BigQuery
already:

| Signal | Why it bears on food access |
|---|---|
| **Poverty rate** | The direct measure of who cannot afford enough food |
| **No-vehicle rate** | If the nearest full grocery store is three miles away and you have no car, you live in a food desert regardless of your income |
| **Public assistance / SNAP rate** | Households already identified as needing food support |

## First, ask what exists

Never hardcode a public table name. The newest ACS tract table has changed underneath us
before—a previous version of our research notes confidently named a table that does not
exist, and it cost a full test cycle. Ask, take the newest, and print what you found.

In [ ]:
def list_tables(dataset_path, contains=None):
    sql = f"""
    SELECT table_name
    FROM `{dataset_path}.INFORMATION_SCHEMA.TABLES`
    {"WHERE table_name LIKE '%" + contains + "%'" if contains else ""}
    ORDER BY table_name DESC
    """
    return [r.table_name for r in client.query(sql).result()]


with step("discover ACS tables"):
    acs_tables = list_tables("bigquery-public-data.census_bureau_acs", "censustract")
    ACS_TABLE = acs_tables[0]

print(f"ACS census-tract tables found: {acs_tables[:6]}")
print(f"Using the newest: {ACS_TABLE}")

# A column NAME is only half a claim. The type is the other half.
type_sql = """
SELECT column_name, data_type
FROM `bigquery-public-data.geo_census_tracts.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'us_census_tracts_national'
  AND column_name IN ('geo_id','internal_point_lat','internal_point_lon','tract_geom')
ORDER BY column_name
"""
print("\nGeometry table column types:")
for r in client.query(type_sql).result():
    print(f"  {r.column_name:<22} {r.data_type}")

Look at those types before you move on, because there is a trap here that has already cost
one debugging cycle on this project.

**`internal_point_lat` and `internal_point_lon` are `STRING`, not `FLOAT64`.** Compare them
to a number and BigQuery fails with `No matching signature for operator BETWEEN`, which does
not mention types at all. Worse is the version that does *not* fail: string comparison sorts
`"9"` above `"10"` without complaining, and you get a silently wrong bounding box.

Every numeric we pull out of a public dataset goes through `SAFE_CAST`. It is free when it
was not needed.

In [ ]:
tracts_sql = f"""
WITH geo AS (
  SELECT
    geo_id,
    SAFE_CAST(internal_point_lat AS FLOAT64) AS tract_lat,
    SAFE_CAST(internal_point_lon AS FLOAT64) AS tract_lon,
    tract_geom
  FROM `bigquery-public-data.geo_census_tracts.us_census_tracts_national`
  WHERE SAFE_CAST(internal_point_lat AS FLOAT64) BETWEEN {LAT_MIN} AND {LAT_MAX}
    AND SAFE_CAST(internal_point_lon AS FLOAT64) BETWEEN {LON_MIN} AND {LON_MAX}
)
SELECT
  g.geo_id,
  g.tract_lat,
  g.tract_lon,
  SAFE_CAST(a.total_pop AS FLOAT64)    AS total_pop,
  SAFE_CAST(a.median_income AS FLOAT64) AS median_income,
  SAFE_DIVIDE(SAFE_CAST(a.poverty AS FLOAT64),
              SAFE_CAST(a.pop_determined_poverty_status AS FLOAT64)) AS poverty_rate,
  SAFE_DIVIDE(SAFE_CAST(a.no_cars AS FLOAT64),
              SAFE_CAST(a.households AS FLOAT64))                    AS no_vehicle_rate,
  SAFE_DIVIDE(SAFE_CAST(a.households_public_asst_or_food_stamps AS FLOAT64),
              SAFE_CAST(a.households AS FLOAT64))                    AS assistance_rate
FROM geo g
JOIN `bigquery-public-data.census_bureau_acs.{ACS_TABLE}` a
  ON a.geo_id = g.geo_id
WHERE SAFE_CAST(a.total_pop AS FLOAT64) > 0
"""

with step("pull census tracts"):
    job = client.query(tracts_sql)
    tracts = job.to_dataframe()
    ACS_BYTES = job.total_bytes_processed

print(f"Census tracts in the {METRO} box: {len(tracts):,}")
print(f"Bytes processed: {ACS_BYTES / 1e6:.1f} MB  (free tier covers 1 TiB/month)")
tracts[["geo_id", "total_pop", "poverty_rate", "no_vehicle_rate", "assistance_rate"]].describe().T

Two ACS quirks are visible in that query, and both are worth knowing.

**There is no poverty rate column.** ACS gives you a count (`poverty`) and a denominator
(`pop_determined_poverty_status`), and they are not the same as `total_pop`—the denominator
excludes people for whom poverty status cannot be determined. Divide the wrong pair and your
rate is wrong in a way that looks plausible.

**`households_public_asst_or_food_stamps` is an OR.** It counts households on public
assistance *or* on food stamps. It over-counts SNAP specifically. If you put "42% of
households receive SNAP" on a slide, that number is not quite what you said it was. Say
"receive public assistance or SNAP" and you are both accurate and more credible.

Now we attach each organization to the tract it sits in. We only have ZIP codes from the IRS,
so this is an approximation—a real deployment would geocode the street address. **Say that
out loud in your demo rather than letting a judge find it.**

In [ ]:
# Approximate: place each organization at the centroid of the tracts in its ZIP.
zip_sql = f"""
SELECT
  z.zip_code,
  AVG(SAFE_CAST(t.internal_point_lat AS FLOAT64)) AS zip_lat,
  AVG(SAFE_CAST(t.internal_point_lon AS FLOAT64)) AS zip_lon
FROM `bigquery-public-data.geo_us_boundaries.zip_codes` z
JOIN `bigquery-public-data.geo_census_tracts.us_census_tracts_national` t
  ON ST_INTERSECTS(z.zip_code_geom, t.internal_point_geo)
WHERE z.state_code = '{STATE}'
GROUP BY z.zip_code
"""

with step("locate organizations"):
    zips = client.query(zip_sql).to_dataframe()
    recipients_final = recipients_final.merge(
        zips, how="left", left_on="ZIP5", right_on="zip_code")

located = recipients_final["zip_lat"].notna().sum()
print(f"Organizations placed on the map: {located:,} of {len(recipients_final):,}")
if located < len(recipients_final) * 0.5:
    print("  NOTE: many organizations are outside this state's ZIP set. That is expected")
    print("  for planted teaching rows, which carry no ZIP.")

# 10. The other side: what is on offer

Real-time commercial surplus is not public data. No grocer publishes "we have 40kg of
chicken thighs expiring Thursday" to an open API, and none ever will—it is commercially
sensitive and it lasts four hours.

So the surplus postings are ours, and there are only a handful. That is deliberate: **the
postings are the query, not the corpus.** You need enough of them to exercise your matcher
across the three tracks, not five hundred.

Each one is written the way a busy person types into a form at the end of a shift, because
that is what your agent will actually receive.

In [ ]:
SURPLUS = [
    # --- Retail & grocer rescue -------------------------------------------------
    ("RETAIL", "10 crates of spinach, harvested yesterday, needs cold storage",
     "Lettuce", "refrigerated", 6),
    ("RETAIL", "mixed dairy - about 60 gallons of milk and some yogurt, 2 days to code date",
     "Milk", "refrigerated", 8),
    ("RETAIL", "roughly 200 loaves of day-old bread and pastries from three store locations",
     "Bread", "pantry", 24),
    ("RETAIL", "prepared sandwiches and salads from the deli case, must move tonight",
     "Leftovers", "refrigerated", 4),
    # --- Farm & agricultural surplus --------------------------------------------
    ("FARM",   "5 tons of potatoes still in the field, we can load pallets if someone collects",
     "Potatoes", "pantry", 72),
    ("FARM",   "half a truckload of blueberries, picked this morning, extremely perishable",
     "Blueberries", "refrigerated", 12),
    ("FARM",   "surplus winter squash, about 30 bins, no refrigeration needed",
     "Squash", "pantry", 120),
    # --- Critical non-food essentials -------------------------------------------
    ("ESSENTIALS", "48 cases of sealed infant formula, 8 months to expiry",
     "Formula", "pantry", 168),
    ("ESSENTIALS", "pallet of diapers and wipes, sizes 1 through 4",
     None, "pantry", 720),
    ("ESSENTIALS", "temperature-sensitive nutritional supplements for a clinic, cold chain required",
     "Formula", "refrigerated", 8),
]

surplus = pd.DataFrame(SURPLUS, columns=[
    "track", "posting_text", "foodkeeper_product", "storage_state", "hours_available"])
surplus["posting_id"] = range(1, len(surplus) + 1)

# Attach a shelf life from FoodKeeper where we can name the product.
lookup = shelf_life.set_index("product_name")
def shelf_hours(row):
    p = row["foodkeeper_product"]
    if not p or p not in lookup.index:
        return None
    rec = lookup.loc[p]
    if isinstance(rec, pd.DataFrame):
        rec = rec.iloc[0]
    return rec["refrigerated_hours"] if row["storage_state"] == "refrigerated" else rec["pantry_hours"]

surplus["shelf_life_hours"] = surplus.apply(shelf_hours, axis=1)
surplus["hours_remaining"] = surplus[["hours_available", "shelf_life_hours"]].min(axis=1)

print(surplus[["posting_id", "track", "posting_text", "hours_remaining"]].to_string(index=False))

Notice what `hours_remaining` is: **the smaller of what the donor says and what the food
science says.** A donor who claims 72 hours for blueberries is optimistic, and the agent
should not take their word for it.

Notice also the row where `foodkeeper_product` is `None`. Diapers are not food and FoodKeeper
has nothing to say about them, so `shelf_life_hours` is null and `hours_remaining` falls back
to the donor's window. Your matcher has to cope with that rather than crash on it.

**And a limitation to own:** the mapping from a free-text posting to a FoodKeeper product is
hardcoded here. In a real system that lookup is itself a matching problem—which, now that
you have seen Section 8, you might notice is exactly the kind of problem you are about to
learn to solve. **That is a genuinely strong add-on** and very few teams will think of it.

# 11. One decision we made deliberately, and why it is different here

Every challenge in this pack touches demographics, and every one carries the same rule:
**data containing race or ethnicity is excluded as a model input and required instead as an
audit of the output.** This is consistent with Google's own responsible-AI guidance, which
states plainly that models trained on race are typically biased.

The reasoning is worth understanding rather than just following. Race genuinely does
correlate with food insecurity—that is well established and you should not pretend
otherwise. But the correlation is a **proxy**. The causal variables are economic and
structural: income, vehicle access, distance to a full grocery store. We can measure those
directly, and they are in your data. Race is a cruder measurement of something we already
have a better measurement of.

And critically: **removing the column does not remove the bias.** Every correlated proxy is
still there. That approach has a name—"fairness through unawareness"—and it does not
work. The remedy is auditing the output, not deleting the input.

## But here is what makes Challenge 2 different

In a challenge built on a trained model, "exclude it as an input" means dropping a column.
There is a column, you drop it, it is gone.

**You have no columns.** Your differentiator embeds *text*. Whatever is in `profile_text`
goes into the vector, and there is nothing to drop afterwards. If a profile says "we serve a
predominantly Black congregation," that attribute is in your embedding permanently and no
amount of column selection will remove it.

So the decision is not what to drop. **It is what goes into the text in the first place.**
And the honest answer is not "nothing sensitive," because that would break the application.

Here is the line we drew, and we think it is the right one:

> **An attribute that describes an operational constraint belongs in the profile.**
> **An attribute used to rank who deserves the food does not.**

*"We serve a halal-observant community and cannot accept pork"* is not a demographic
flourish. It is a hard matching constraint, and a broker that ignores it ships a pallet that
gets thrown away. The same is true of language, of infant nutrition, and of medically
restricted diets. **Excluding those would produce a worse and less respectful system, not a
fairer one.**

What does not belong is demographic composition used as a **priority signal**—"this
neighbourhood is X% Y, therefore rank it higher." That belongs in the audit, after the fact,
exactly as in every other challenge in this pack.

## What the audit means here, concretely

Three steps, about twenty minutes, and most teams will skip it:

1. **Run your matcher across all ten surplus postings** and collect the recipients it chose.
2. **Look up the tracts those organizations sit in** and pull their demographics from
   `tract_demographics`.
3. **Compare them to the metro as a whole.** Are your chosen recipients in poorer tracts than
   average, or richer? Higher or lower vehicle access?

Then say the answer out loud. **Did the food go where the need is, or where the loading docks
are?**

Both answers are worth having. If your matches skew toward high-poverty, low-vehicle tracts,
you have evidence your system finds real need—say so, with numbers. If they skew the other
way, you have found something more interesting: **your matcher may be optimising for logistics
capability, because big organizations with walk-in freezers write more detailed profiles and
sit in industrial areas.** That is a real and subtle failure mode, it is probably happening,
and presenting it honestly will land far better with judges than a slide claiming everything
worked.

# 12. Load into BigQuery

Four tables, all in your project, all replaceable so you can re-run this notebook safely.

In [ ]:
def load_table(df, table_name, description=""):
    table_id = f"{PROJECT_ID}.{DATASET}.{table_name}"
    job_config = bigquery.LoadJobConfig(
        write_disposition="WRITE_TRUNCATE",     # safe to re-run
        autodetect=True,
    )
    client.load_table_from_dataframe(df, table_id, job_config=job_config).result()
    tbl = client.get_table(table_id)
    if description:
        tbl.description = description
        client.update_table(tbl, ["description"])
    print(f"  {table_name:<22} {tbl.num_rows:>7,} rows   {len(tbl.schema)} columns")
    return tbl


RECIPIENT_COLS = [
    "recipient_id", "EIN", "NAME", "STREET", "CITY", "STATE", "ZIP5", "NTEE_CD",
    "archetype", "archetype_label", "cold_storage", "frozen_storage", "loading_dock",
    "refrigerated_transport", "max_pallets", "pickup_window_hours", "weekly_households",
    "populations_served", "service_days", "profile_text", "profile_source",
    "zip_lat", "zip_lon",
]

with step("load BigQuery tables"):
    out = recipients_final[[c for c in RECIPIENT_COLS if c in recipients_final.columns]].copy()
    out = out.rename(columns={"EIN": "ein", "NAME": "name", "STREET": "street",
                              "CITY": "city", "STATE": "state", "ZIP5": "zip_code",
                              "NTEE_CD": "ntee_code", "zip_lat": "latitude",
                              "zip_lon": "longitude"})
    print("Loading:")
    load_table(out, "recipients",
               "Recipient organizations. Names, addresses and NTEE codes are REAL "
               "(IRS Exempt Organizations BMF, US federal public domain). Operational "
               "attributes and profile_text are GENERATED - see notebook Section 7.")
    load_table(surplus, "surplus_postings",
               "Synthetic surplus offers across three tracks. These are the query side.")
    load_table(shelf_life, "shelf_life",
               "USDA FSIS FoodKeeper shelf life, CC0. Consumer home-storage guidance, "
               "not a commercial cold-chain model.")
    load_table(tracts.drop(columns=[c for c in ["tract_geom"] if c in tracts.columns]),
               "tract_demographics",
               "ACS census-tract demographics for the metro. Poverty, vehicle access, "
               "and public assistance rates. No race or ethnicity columns - see Section 11.")

Let's ask BigQuery something we could not have asked before: **what is the storage capacity
of this city's food-assistance network?**

In [ ]:
%%bigquery
SELECT
  cold_storage,
  COUNT(*)                      AS organizations,
  SUM(weekly_households)        AS households_reached_weekly,
  ROUND(AVG(max_pallets), 1)    AS avg_pallet_capacity,
  COUNTIF(refrigerated_transport) AS can_collect_chilled
FROM `a4i_food.recipients`
GROUP BY cold_storage
ORDER BY organizations DESC

That table is the challenge in miniature. A large share of the network has **no refrigeration
at all**, and a further group has only a domestic fridge. The pallet of spinach can only go to
a fraction of these organizations—and finding which fraction, quickly, is the job.

If you had matched on category alone, every one of these rows is "a food bank."

# 13. Validate before you match

Every check below queries the **loaded BigQuery tables**, not the dataframes still in memory.
Those are different things, and the difference is where load errors hide.

The framing worth internalising: **the errors that hurt you are the ones that do not raise.**
An empty query succeeds. A silently-nulled column succeeds. A join that matches nothing
returns zero rows and no error at all.

In [ ]:
CHECKS = []
def check(name, passed, detail=""):
    CHECKS.append((name, bool(passed), str(detail)))

def q1(sql):
    return client.query(sql).to_dataframe().iloc[0]

DS = f"{PROJECT_ID}.{DATASET}"

# ---------------------------------------------------------------- recipients
r = q1(f"""
SELECT
  COUNT(*)                                   AS n,
  COUNT(DISTINCT recipient_id)               AS ids,
  COUNTIF(profile_text IS NULL OR LENGTH(profile_text) < 40) AS thin_profiles,
  COUNT(DISTINCT archetype)                  AS archetypes,
  COUNTIF(profile_source = 'generated')      AS generated,
  COUNTIF(profile_source = 'planted')        AS planted,
  COUNTIF(cold_storage IN ('walk-in','commercial','medical','vehicle')) AS real_cold,
  AVG(LENGTH(profile_text))                  AS avg_len,
  COUNT(DISTINCT profile_text)               AS distinct_profiles
FROM `{DS}.recipients`
""")
check("recipients: has rows",         r.n > 100,                f"{int(r.n):,} organizations")
check("recipients: ids unique",       r.ids == r.n,             f"{int(r.ids):,} distinct ids")
check("recipients: profiles usable",  r.thin_profiles == 0,     f"{int(r.thin_profiles)} too short")
check("recipients: archetype spread", r.archetypes >= 8,        f"{int(r.archetypes)} archetypes")
check("recipients: planted present",  r.planted >= 8,           f"{int(r.planted)} planted cases")
check("recipients: cold storage mix", 0 < r.real_cold < r.n,    f"{int(r.real_cold)} with real refrigeration")

# THE one that decides whether vector search can work at all.
variety = r.distinct_profiles / r.n
check("recipients: profiles varied",  variety > 0.9,
      f"{variety:.1%} distinct ({int(r.distinct_profiles):,} of {int(r.n):,})")

# No personal-name column may survive.
cols = [c.name for c in client.get_table(f"{DS}.recipients").schema]
check("recipients: no ICO column",    "ICO" not in cols and "ico" not in cols,
      "in-care-of name excluded")

# ---------------------------------------------------------------- shelf life
s = q1(f"""
SELECT
  COUNT(*)                                  AS n,
  COUNTIF(refrigerated_hours IS NOT NULL)   AS with_fridge,
  COUNTIF(refrigerated_hours = 0)           AS zero_hours,
  COUNTIF(refrigerated_note = 'not_recommended') AS not_recommended,
  MAX(refrigerated_hours)                   AS max_hours
FROM `{DS}.shelf_life`
""")
check("shelf life: has rows",         s.n > 500,                f"{int(s.n):,} products")
check("shelf life: some parsed",      s.with_fridge > 50,       f"{int(s.with_fridge):,} with a figure")
check("shelf life: no false zeros",   s.zero_hours == 0,
      "sentinels became notes, not zeros")

# ---------------------------------------------------------------- surplus
p = q1(f"""
SELECT COUNT(*) AS n,
       COUNT(DISTINCT track) AS tracks,
       COUNTIF(hours_remaining IS NULL) AS no_clock
FROM `{DS}.surplus_postings`
""")
check("surplus: has rows",            p.n >= 8,                 f"{int(p.n)} postings")
check("surplus: all three tracks",    p.tracks == 3,            f"{int(p.tracks)} tracks")

# ---------------------------------------------------------------- demographics
t = q1(f"""
SELECT
  COUNT(*)                                            AS n,
  COUNTIF(poverty_rate < 0 OR poverty_rate > 1)       AS bad_poverty,
  COUNTIF(no_vehicle_rate < 0 OR no_vehicle_rate > 1) AS bad_vehicle,
  ROUND(AVG(poverty_rate), 4)                         AS avg_poverty,
  COUNTIF(LENGTH(geo_id) != 11)                       AS bad_geoid
FROM `{DS}.tract_demographics`
""")
check("tracts: has rows",             t.n > 20,                 f"{int(t.n):,} tracts")
check("tracts: rates in range",       t.bad_poverty == 0 and t.bad_vehicle == 0,
      f"mean poverty {t.avg_poverty:.1%}")
check("tracts: geo_id well formed",   t.bad_geoid == 0,         "11-digit tract ids")

tcols = [c.name.lower() for c in client.get_table(f"{DS}.tract_demographics").schema]
banned = [c for c in tcols if any(w in c for w in
          ("black", "white", "hispanic", "asian", "race", "ethnic"))]
check("tracts: no race columns",      not banned,               banned or "none present")

# ---------------------------------------------------------------- cross-table
x = q1(f"""
SELECT
  (SELECT COUNT(*) FROM `{DS}.recipients` WHERE latitude IS NOT NULL) AS located,
  (SELECT COUNT(*) FROM `{DS}.surplus_postings` s
     JOIN `{DS}.shelf_life` f ON f.product_name = s.foodkeeper_product) AS joined
""")
check("cross: organizations located", x.located > 0,            f"{int(x.located):,} with coordinates")
check("cross: postings resolve",      x.joined > 0,             f"{int(x.joined)} postings matched to shelf life")

print(f"{len(CHECKS)} checks run.")

In [ ]:
width = max(len(name) for name, _, _ in CHECKS)
passed = sum(1 for _, ok, _ in CHECKS if ok)

print("=" * (width + 40))
print(f"{'VALIDATION':<{width}}   RESULT   DETAIL")
print("=" * (width + 40))
for name, ok, detail in CHECKS:
    print(f"{name:<{width}}   {'PASS  ' if ok else 'FAIL >>'}  {detail}")
print("=" * (width + 40))
print(f"{passed} of {len(CHECKS)} checks passed")

failures = [n for n, ok, _ in CHECKS if not ok]
if failures:
    print("\nInvestigate before you build on this:")
    for n in failures:
        print(f"  - {n}")
else:
    print("\nAll clear. Your tables are sound. Go build.")

# 14. Framing your matcher

This is where the notebook stops and your team starts.

## What you are building

Embed every recipient's `profile_text` once. Then, for each surplus posting, embed the
posting and find the nearest recipients in that vector space. That is `VECTOR_SEARCH`, and it
is your required differentiator: **your agent must call the search, not filter on keywords or
categories.**

## The four traps, so you do not lose forty minutes to them

We have burned these already so you do not have to. None of them teaches you anything about
food equity.

**1. `AI.EMBED` returns a STRUCT, not an array.** It gives you
`STRUCT<result ARRAY<FLOAT64>, status STRING>`. `VECTOR_SEARCH` wants the array. Pull
`.result` out when you materialise the table.

**2. Filter the failures before you search.** Rows where embedding failed carry a non-empty
`status`. Keep them and your search silently ranks garbage. `WHERE status = ''` is the filter.

**3. Do not bother with `CREATE VECTOR INDEX`.** A vector index does not populate below about
10 MB of data, and 500 short profiles is nowhere near that. The index will appear to build and
will silently stay empty. Pass `options => '{"use_brute_force":true}'` instead—at this scale
brute force is instant, and it is the correct call rather than a workaround. Building the index
is worth demonstrating once, to show you know what it is for.

**4. Your connection region must match your dataset region.** A dataset in `US` with a
connection in `us-central1` fails with an error that never mentions regions.

## The shape

```sql
-- Embed the corpus once. AI.EMBED names the model inline - no CREATE MODEL needed.
CREATE OR REPLACE TABLE `a4i_food.recipient_embeddings` AS
SELECT
  * EXCEPT (embedded),
  embedded.result AS embedding
FROM (
  SELECT r.*, AI.EMBED(r.profile_text, endpoint => 'text-embedding-005') AS embedded
  FROM `a4i_food.recipients` r
)
WHERE embedded.status = '';

-- Then search it, once per posting.
SELECT base.name, base.cold_storage, base.max_pallets, distance
FROM VECTOR_SEARCH(
  TABLE `a4i_food.recipient_embeddings`, 'embedding',
  (SELECT AI.EMBED('10 crates of spinach, harvested yesterday, needs cold storage',
                   endpoint => 'text-embedding-005').result AS embedding),
  top_k => 10,
  options => '{"use_brute_force":true}');
```

That is the mechanism. **It is not the challenge.**

## What is actually yours to decide

Run that query and look hard at the results, because the top hit is probably wrong.

Remember the planted decoy from Section 7—the organization that desperately wants fresh
greens and has **no refrigeration whatsoever.** Semantically it is a superb match. Operationally
it is a pallet of spoiled spinach. Vector search will rank it near the top and it will be
confidently, uselessly wrong.

**Vector search is a candidate generator, not an answer.** The four decisions that separate
teams all live after it:

| Decision | The question |
|---|---|
| **What goes into the embedded text** | We embedded `profile_text`. Should you? Would adding capacity or hours help—or would it flood the vector with numbers and wash out the meaning? |
| **How you compose the query** | We embedded the raw posting. Should you enrich it first—"needs refrigeration, 6 hours, 10 crates"—and does that help or hurt? |
| **What you do after the search** | Storage compatibility, pallet capacity, pickup window against `hours_remaining`, service days. Filter? Re-rank? Explain? |
| **How you rank what survives** | Two viable recipients—do you send it to the bigger one, the closer one, or the one in the higher-need tract? **There is no right answer and judges know it. They want to hear you choose on purpose.** |

## Your output artifact

Whatever else your agent does, it should produce a match a coordinator could act on:

- **Which organization**, by name, with its address
- **Why them**—what in their profile made them the fit, in a sentence a human would accept
- **What they can actually take**—matched against the pallet count and storage on offer
- **The clock**—how long the food has, and whether this recipient can collect inside it
- **The drafted outreach message**, ready to send

## What we deliberately did not build

There is no `AI.EMBED` call and no `VECTOR_SEARCH` in this notebook. That is the differentiator
and it is yours. We built the on-ramp: real organizations, an honest corpus, a clock, and a
need signal—all validated. **The vehicle is your work.**

In [ ]:
print(f"Project        : {PROJECT_ID}")
print(f"Dataset        : {DATASET} ({LOCATION})")
print(f"Metro          : {METRO}, {STATE}")
print(f"Embedding model: {EMBED_ENDPOINT}")
print()
print("Your tables:")
for t in client.list_tables(f"{PROJECT_ID}.{DATASET}"):
    tbl = client.get_table(t)
    print(f"  {tbl.table_id:<22} {tbl.num_rows:>7,} rows")
print()
print("Next: embed recipients.profile_text, then VECTOR_SEARCH it.")
print("See Section 14 for the syntax and the four traps.")

# Appendix: diagnostic summary

Run this cell and paste what it prints when you ask a coach for help, or when you report a
problem with the data. One block beats twenty screenshots.

In [ ]:
print("=" * 68)
print("A4I CHALLENGE 2 - DIAGNOSTIC SUMMARY")
print("=" * 68)
print(f"Project         : {PROJECT_ID}")
print(f"Dataset         : {DATASET} ({LOCATION})")
print(f"Metro / state   : {METRO} / {STATE}")
print(f"Embed endpoint  : {EMBED_ENDPOINT}")
print("-" * 68)
print("DISCOVERED NAMES")
print(f"  ACS table          : {globals().get('ACS_TABLE', 'not reached')}")
print(f"  IRS source         : {globals().get('EO_URL', 'not reached')}")
acsb = globals().get("ACS_BYTES")
print(f"  ACS bytes scanned  : {acsb / 1e6:.1f} MB" if acsb else "  ACS bytes: not reached")
print("-" * 68)
try:
    print(f"IRS rows for state  : {len(eo_all):,}")
    print(f"After NTEE filter   : {len(eo):,}")
    print(f"Corpus size         : {len(recipients_final):,}")
    print(f"  generated         : {(recipients_final.profile_source == 'generated').sum():,}")
    print(f"  planted           : {(recipients_final.profile_source == 'planted').sum():,}")
    print(f"Archetypes used     : {recipients_final.archetype.nunique()}")
    print(f"Distinct profiles   : {recipients_final.profile_text.nunique():,}")
    print(f"Mean profile chars  : {recipients_final.profile_text.str.len().mean():.0f}")
    print(f"Real profiles seen  : {len(real_profiles):,}")
    print(f"FoodKeeper products : {len(shelf_life):,}")
    print(f"Census tracts       : {len(tracts):,}")
except Exception as exc:                              # noqa: BLE001
    print(f"A stage did not complete: {exc}")
print("-" * 68)
print("LOADED TABLES")
try:
    for t in client.list_tables(f"{PROJECT_ID}.{DATASET}"):
        tbl = client.get_table(t)
        print(f"{tbl.table_id}: {tbl.num_rows:,} rows")
        print(f"    {', '.join(f.name for f in tbl.schema)}")
except Exception as exc:                              # noqa: BLE001
    print(f"Could not list tables: {exc}")
print("-" * 68)
print("VALIDATION")
try:
    for name, ok, detail in CHECKS:
        print(f"  {'PASS' if ok else 'FAIL'}  {name}: {detail}")
except NameError:
    print("  validation section not reached")
print("-" * 68)
print("TIMING")
for name, secs in STEP_TIMES.items():
    print(f"  {name:<30} {secs:>7.1f}s")
print(f"  {'TOTAL WALL CLOCK':<30} {time.time() - NOTEBOOK_START:>7.1f}s")
print("=" * 68)